In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import os
import time
from typing import Sequence, Tuple
import numpy as np
import jax
import jax.numpy as jnp
from jaxued.environments.underspecified_env import EnvParams, EnvState, UnderspecifiedEnv
from jaxued.utils import compute_max_mean_returns_epcount, compute_max_mean_returns_epcount_w_idxs
import optax
from flax import struct
from flax.training.train_state import TrainState as BaseTrainState
import flax.linen as nn
from flax.linen.initializers import constant, orthogonal
import distrax
import orbax.checkpoint as ocp
import wandb
from jaxued.environments.maze.env_editor import MazeEditor, Observation, ObservedMazeEditor, ObservedMazeEditorWithGoal
from jaxued.linen import ResetRNN
from jaxued.environments import Maze, MazeRenderer, ObservedMazeRenderer
from jaxued.environments.maze import Level, ObservedLevel
from jaxued.wrappers import AutoReplayWrapper
import chex

import logging
import hydra
from omegaconf import DictConfig, OmegaConf
import matplotlib.pyplot as plt

In [3]:
from omegaconf import OmegaConf
from hydra import initialize, compose


def load_hydra_config(config_path, config_name):
    # Initialize the Hydra context
    with initialize(config_path=config_path):
        # Compose the configuration
        cfg = compose(config_name=config_name)
        return cfg
    
config_path = "config"  # path to the directory containing the config file
config_name = "main_paired"  # name of the config file without the extension

config = load_hydra_config(config_path, config_name)

if config["num_env_steps"] is not None:
    config["num_updates"] = config["num_env_steps"] // (config["num_train_envs"] * config["num_steps"])

if config['mode'] == 'eval':
    os.environ['WANDB_MODE'] = 'disabled'
    

/tmp/ipykernel_4142/2007320477.py:7: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize(config_path=config_path):


In [4]:
# region PPO helper functions    
@struct.dataclass
class TrainState:
    update_count: int
    pro_train_state: BaseTrainState
    ant_train_state: BaseTrainState
    adv_train_state: BaseTrainState


def compute_gae(
    gamma: float,
    lambd: float,
    last_value: chex.Array,
    values: chex.Array,
    rewards: chex.Array,
    dones: chex.Array,
    traj_idxs: chex.Array
) -> Tuple[chex.Array, chex.Array]:
    """This takes in arrays of shape (NUM_STEPS, NUM_ENVS) and returns the advantages and targets.

    Args:
        gamma (float): 
        lambd (float): 
        last_value (chex.Array):  Shape (NUM_ENVS)
        values (chex.Array): Shape (NUM_STEPS, NUM_ENVS)
        rewards (chex.Array): Shape (NUM_STEPS, NUM_ENVS)
        dones (chex.Array): Shape (NUM_STEPS, NUM_ENVS)

    Returns:
        Tuple[chex.Array, chex.Array]: advantages, targets; each of shape (NUM_STEPS, NUM_ENVS)
    """
    def compute_gae_at_timestep(carry, x):
        gae, next_value, cur_idx = carry
        traj_started = cur_idx <= traj_idxs
    
        value, reward, done = x
        delta = reward + gamma * next_value * (1 - done) * traj_started - value
        gae = delta + gamma * lambd * (1 - done) * gae
        return (gae, jnp.where(traj_started, value, next_value), cur_idx - 1), gae

    _, advantages = jax.lax.scan(
        compute_gae_at_timestep,
        (jnp.zeros_like(last_value), last_value, values.shape[0]),
        (values, rewards, dones),
        reverse=True,
        unroll=16,
    )
    return advantages, advantages + values


class ActorCritic(nn.Module):
    action_dim: Sequence[int]
    
    @nn.compact
    def __call__(self, inputs, hidden):
        obs, dones = inputs
        
        img_embed = nn.Conv(16, kernel_size=(3, 3), strides=(1, 1), padding="VALID")(obs.image)
        img_embed = img_embed.reshape(*img_embed.shape[:-3], -1)
        img_embed = nn.relu(img_embed)
        
        dir_embed = jax.nn.one_hot(obs.agent_dir, 4)
        dir_embed = nn.Dense(5, kernel_init=orthogonal(np.sqrt(2)), bias_init=constant(0.0), name="scalar_embed")(dir_embed)
        
        embedding = jnp.append(img_embed, dir_embed, axis=-1)

        hidden, embedding = ResetRNN(nn.OptimizedLSTMCell(features=256))((embedding, dones), initial_carry=hidden)

        actor_mean = nn.Dense(32, kernel_init=orthogonal(2), bias_init=constant(0.0), name="actor0")(embedding)
        actor_mean = nn.tanh(actor_mean)
        actor_mean = nn.Dense(self.action_dim, kernel_init=orthogonal(0.01), bias_init=constant(0.0), name="actor1")(actor_mean)
        pi = distrax.Categorical(logits=actor_mean)

        critic = nn.Dense(32, kernel_init=orthogonal(2), bias_init=constant(0.0), name="critic0")(embedding)
        critic = nn.relu(critic)
        critic = nn.Dense(1, kernel_init=orthogonal(1.0), bias_init=constant(0.0), name="critic1")(critic)

        return hidden, pi, jnp.squeeze(critic, axis=-1)
    
    @staticmethod
    def initialize_carry(batch_dims):
        return nn.OptimizedLSTMCell(features=256).initialize_carry(jax.random.PRNGKey(0), (*batch_dims, 256))


class AdversaryActorCritic(nn.Module):
    # The adversary's network architecture
    action_dim: Sequence[int]
    max_timesteps: int = 50
    
    @nn.compact
    def __call__(self, inputs: Tuple[Observation, chex.Array], hidden):
        obs, dones = inputs
        
        img_embed = nn.Conv(128, kernel_size=(3, 3), strides=(1, 1), padding="VALID")(jnp.concatenate((obs.image, jnp.expand_dims(obs.observation_map, axis=-1), obs.agent_observations), axis=-1))
        img_embed = img_embed.reshape(*img_embed.shape[:-3], -1)
        img_embed = nn.relu(img_embed)
        
        time_value = nn.Embed(self.max_timesteps + 1, 10, name="time_embed", embedding_init=orthogonal(1.0))(jnp.clip(obs.time, None, self.max_timesteps))
        random_z_value = obs.random_z
        embedding = jnp.concatenate((img_embed, time_value, random_z_value, obs.place_goal[..., None]), axis=-1)

        hidden, embedding = ResetRNN(nn.OptimizedLSTMCell(features=256))((embedding, dones), initial_carry=hidden)

        actor_mean = nn.Dense(32, kernel_init=orthogonal(2), bias_init=constant(0.0), name="actor0")(embedding)
        actor_mean = nn.relu(actor_mean)
        actor_mean = nn.Dense(self.action_dim, kernel_init=orthogonal(0.01), bias_init=constant(0.0), name="actor1")(actor_mean)

        # Mask out this
        actor_mean = jnp.where(obs.action_mask, actor_mean, -jnp.inf)
        pi = distrax.Categorical(logits=actor_mean)

        critic = nn.Dense(32, kernel_init=orthogonal(2), bias_init=constant(0.0), name="critic0")(embedding)
        critic = nn.relu(critic)
        critic = nn.Dense(1, kernel_init=orthogonal(1.0), bias_init=constant(0.0), name="critic1")(critic)

        return hidden, pi, jnp.squeeze(critic, axis=-1)
    
    @staticmethod
    def initialize_carry(batch_dims):
        return nn.OptimizedLSTMCell(features=256).initialize_carry(jax.random.PRNGKey(0), (*batch_dims, 256))

In [5]:
env = Maze(max_height=13, max_width=13, agent_view_size=5, normalize_obs=True)
adv_env = ObservedMazeEditorWithGoal(env, random_z_dimensions=config['adv_random_z_dimension'], zero_out_random_z=config['adv_zero_out_random_z'])
adv_env_renderer = ObservedMazeRenderer(env, tile_size=8)
env_renderer = MazeRenderer(env, tile_size=8)
env = AutoReplayWrapper(env)
env_params = env.default_params
adv_env_params = adv_env.default_params

rng = jax.random.PRNGKey(14532)
rng, _rng = jax.random.split(rng)

In [6]:
def sample_empty_level():
    w, h = env._env.max_width, env._env.max_height
    return ObservedLevel(
        wall_map=jnp.zeros((h, w), dtype=jnp.bool_),
        observation_map=jnp.zeros((h, w), dtype=jnp.bool_),
        width=w,
        height=h,
        
        # These values don't matter, as the adversary overwrites them.
        goal_pos=jnp.array([0, 0], dtype=jnp.uint32),
        agent_pos=jnp.array([1, 1], dtype=jnp.uint32),
        agent_dir=jnp.array(0, dtype=jnp.uint8),
        goal_placed=jnp.array(False, dtype=jnp.bool_),
    )

def create_train_state(rng):
    def create_inner_train_state(rng, env, env_params, network_cls, prefix, network_kws={}):
        def linear_schedule(count):
            frac = (
                1.0
                - (count // (config[f"{prefix}num_minibatches"] * config[f"{prefix}epoch_ppo"]))
                / config["num_updates"]
            )
            return config[f"{prefix}lr"] * frac
        obs, _ = env.reset_to_level(rng, sample_empty_level(), env_params)
        obs = jax.tree.map(
            lambda x: jnp.repeat(jnp.repeat(x[None, ...], config["num_train_envs"], axis=0)[None, ...], 256, axis=0),
            obs,
        )
        init_x = (obs, jnp.zeros((256, config["num_train_envs"])))
        network = network_cls(env.action_space(env_params).n, **network_kws)
        network_params = network.init(rng, init_x, network_cls.initialize_carry((config["num_train_envs"],)))
        tx = optax.chain(
            optax.clip_by_global_norm(config[f"{prefix}max_grad_norm"]),
            optax.adam(learning_rate=linear_schedule, eps=1e-5),
            # optax.adam(learning_rate=config[f"{prefix}lr"], eps=1e-5),
        )
        return BaseTrainState.create(
            apply_fn=network.apply,
            params=network_params,
            tx=tx,
        )
    rng_pro, rng_ant, rng_adv = jax.random.split(rng, 3)
    return TrainState(
        update_count = 0,
        pro_train_state = create_inner_train_state(rng_pro, env, env_params, ActorCritic, "student_"),
        ant_train_state = create_inner_train_state(rng_ant, env, env_params, ActorCritic, "student_"),
        adv_train_state = create_inner_train_state(rng_adv, adv_env, adv_env_params, AdversaryActorCritic, "adv_", network_kws={"max_timesteps": config["adv_num_steps"]})
    )

In [7]:
rng, _rng = jax.random.split(rng)
train_state = create_train_state(_rng)

# Checkpointing

In [8]:
def setup_checkpointing(config: dict, train_state: TrainState, env: UnderspecifiedEnv, env_params: EnvParams) -> ocp.CheckpointManager:
    """This takes in the train state and config, and returns an orbax checkpoint manager.
        It also saves the config in `checkpoints/run_name/seed/config.json`

    Args:
        config (dict): 
        train_state (TrainState): 
        env (UnderspecifiedEnv): 
        env_params (EnvParams): 

    Returns:
        ocp.CheckpointManager: 
    """
    overall_save_dir = os.path.join(os.getcwd(), "checkpoints", f"{config['run_name']}", str(config['seed']))
    os.makedirs(overall_save_dir, exist_ok=True)
    
    # save the config
    config_dict = OmegaConf.to_container(config, resolve=True)
    with open(os.path.join(overall_save_dir, 'config.json'), 'w+') as f:
        f.write(json.dumps(config_dict, indent=True))
    
    checkpoint_manager = ocp.CheckpointManager(
        os.path.join(overall_save_dir, 'models'),
        options=ocp.CheckpointManagerOptions(
            save_interval_steps=config['checkpoint_save_interval'],
            max_to_keep=config['max_number_of_checkpoints'],
            enable_async_checkpointing=False,
        )
    )

    return checkpoint_manager

In [9]:
import time

In [44]:
checkpoint_manager = setup_checkpointing(config, train_state, env, env_params)
checkpoint_manager

In [138]:
options = ocp.CheckpointManagerOptions()
with ocp.CheckpointManager(
  ocp.test_utils.erase_and_create_empty('/tmp/ckpt2/'),
  options=options,
) as mngr:
  mngr.save(0, args=ocp.args.StandardSave(train_state))

AttributeError: module 'orbax.checkpoint.test_utils' has no attribute 'erase_and_create_empty'

In [24]:
i=9

In [45]:
for _ in range(30):
    checkpoint_manager.save(i, args=ocp.args.StandardSave(train_state))
    checkpoint_manager.wait_until_finished()
    i += 1

# overall_save_dir = os.path.join(os.getcwd(), "checkpoints", f"{config['run_name']}", str(config['seed']), 'models')
# last_checkpoint = os.listdir(overall_save_dir)[-1]
# last_checkpoint[:-38]

# os.rename(os.path.join(overall_save_dir, last_checkpoint), os.path.join(overall_save_dir, last_checkpoint[:-38]))

PermissionError: [Errno 13] Permission denied: '/workspace/ued/obs_generation/checkpoints/optimal_obs_gen/98713298/models/31.orbax-checkpoint-tmp-1729701886374513' -> '/workspace/ued/obs_generation/checkpoints/optimal_obs_gen/98713298/models/31'

In [89]:
overall_save_dir = os.path.join(os.getcwd(), "checkpoints", f"{config['run_name']}", str(config['seed']), 'models')
last_checkpoint = os.listdir(overall_save_dir)[-1]
last_checkpoint[:-38]

os.rename(os.path.join(overall_save_dir, last_checkpoint), os.path.join(overall_save_dir, last_checkpoint[:-38]))

In [28]:
print("Current user:", os.geteuid())

Current user: 0


In [42]:
for i in range(50):
    os.mkdir("test_folder/tst_dir_tmp")
    f = open("test_folder/tst_dir_tmp/new_fimyfile.txt", "x")
    os.rename("test_folder/tst_dir_tmp", f"test_folder/tst_dir_{i}")

PermissionError: [Errno 13] Permission denied: 'test_folder/tst_dir_tmp' -> 'test_folder/tst_dir_0'

In [59]:
os.rename("/workspace/ued/obs_generation/checkpoints/optimal_obs_gen/98713298/models/0.orbax-checkpoint-tmp-1729610864953653", "/workspace/ued/obs_generation/checkpoints/optimal_obs_gen/98713298/models/0")

In [25]:
checkpoint_directory = "checkpoints/None/41234912"
checkpoint_to_eval = 5

rng_init, rng_eval = jax.random.split(jax.random.PRNGKey(10000))

with open(os.path.join(checkpoint_directory, 'config.json')) as f: config = json.load(f)
checkpoint_manager = ocp.CheckpointManager(os.path.join(os.getcwd(), checkpoint_directory, 'models'), item_handlers=ocp.StandardCheckpointHandler())

train_state_og: TrainState = create_train_state(rng_init)
step = checkpoint_manager.latest_step() if checkpoint_to_eval == -1 else checkpoint_to_eval

loaded_checkpoint = checkpoint_manager.restore(step)
params = loaded_checkpoint['pro_train_state']['params']
train_state = train_state_og.replace(pro_train_state=train_state_og.pro_train_state.replace(params=params))

FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints/None/41234912/config.json'

In [12]:
train_state.pro_train_state.params

{'params': {'Conv_0': {'bias': Array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],      dtype=float32),
   'kernel': Array([[[[-0.06737429,  0.01656518, -0.14110538, -0.09687821,
              0.04443626, -0.16747007,  0.35193598, -0.1923744 ,
              0.2620624 , -0.26773888, -0.17901917,  0.28193834,
              0.0449177 , -0.200579  ,  0.32517377,  0.34502554],
            [-0.09988217,  0.24992236,  0.22352104,  0.12569103,
             -0.15182489, -0.20428655,  0.07779282, -0.01525732,
             -0.02881442,  0.12000758, -0.05932071,  0.09204167,
             -0.05880543, -0.02360057,  0.13595569, -0.06257562],
            [ 0.09285276,  0.28928778,  0.02172172, -0.25340858,
              0.25395966, -0.1352421 ,  0.08819814, -0.20431793,
             -0.08412459, -0.3518628 , -0.01757716,  0.11302673,
              0.00613313,  0.19597083, -0.04922705, -0.24402371]],
   
           [[ 0.06391704,  0.00655996,  0.17073064, -0.06965522,
             

In [13]:
train_state.pro_train_state.params

{'params': {'Conv_0': {'bias': Array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],      dtype=float32),
   'kernel': Array([[[[-0.06737429,  0.01656518, -0.14110538, -0.09687821,
              0.04443626, -0.16747007,  0.35193598, -0.1923744 ,
              0.2620624 , -0.26773888, -0.17901917,  0.28193834,
              0.0449177 , -0.200579  ,  0.32517377,  0.34502554],
            [-0.09988217,  0.24992236,  0.22352104,  0.12569103,
             -0.15182489, -0.20428655,  0.07779282, -0.01525732,
             -0.02881442,  0.12000758, -0.05932071,  0.09204167,
             -0.05880543, -0.02360057,  0.13595569, -0.06257562],
            [ 0.09285276,  0.28928778,  0.02172172, -0.25340858,
              0.25395966, -0.1352421 ,  0.08819814, -0.20431793,
             -0.08412459, -0.3518628 , -0.01757716,  0.11302673,
              0.00613313,  0.19597083, -0.04922705, -0.24402371]],
   
           [[ 0.06391704,  0.00655996,  0.17073064, -0.06965522,
             